# Silver Layer: Green Taxi Validation

Validates the completed Silver `green_taxi` table through reconciliation, required-field checks, categorical checks, duplicate profiling, and the final PASS/FAIL summary.

**Prerequisite:** Run `01_silver_green_taxi.ipynb` first.


## 2. Green Taxi Validation

Validation checks for the Silver `green_taxi` table:
- Row count and source file count
- Trip-distance quality checks, including zero, extreme, and negative values
- Trip-distance nullification verification
- Out-of-range datetime checks
- Invalid trip-duration checks (dropoff before pickup)
- Trip-duration calculation and nullification verification
- Systematic NULL pattern checks
- Sample flagged records
- Schema and column verification

In [0]:
%sql
-- Verify the Silver table was created successfully
SELECT 
  COUNT(*) AS total_rows,
  COUNT(DISTINCT source_file) AS source_files,

  -- Trip distance handling
  SUM(CASE 
        WHEN dq_zero_trip_distance THEN 1 
        ELSE 0 
      END) AS rows_with_zero_trip_distance,

  SUM(CASE 
        WHEN dq_extreme_trip_distance THEN 1 
        ELSE 0 
      END) AS rows_with_extreme_trip_distance,

  SUM(CASE 
        WHEN dq_negative_trip_distance THEN 1 
        ELSE 0 
      END) AS rows_with_negative_trip_distance,

  -- Verify invalid trip distances are nullified
  SUM(CASE 
        WHEN trip_distance IS NULL 
         AND dq_extreme_trip_distance 
        THEN 1 
        ELSE 0 
      END) AS extreme_trip_distance_nullified,

  SUM(CASE 
        WHEN trip_distance IS NULL 
         AND dq_negative_trip_distance 
        THEN 1 
        ELSE 0 
      END) AS negative_trip_distance_nullified,

  -- Datetime validation
  SUM(CASE 
        WHEN dq_out_of_range_datetime THEN 1 
        ELSE 0 
      END) AS rows_with_out_of_range_datetime,

  -- Trip duration validation
  SUM(CASE 
        WHEN dq_invalid_trip_duration THEN 1 
        ELSE 0 
      END) AS rows_with_invalid_trip_duration,

  SUM(CASE 
        WHEN dq_invalid_trip_duration
         AND trip_duration_minutes IS NOT NULL
        THEN 1 
        ELSE 0 
      END) AS invalid_duration_with_value,

  SUM(CASE 
        WHEN dq_invalid_trip_duration
         AND trip_duration_minutes IS NULL
        THEN 1 
        ELSE 0 
      END) AS invalid_duration_nullified,

  -- Systematic NULL pattern preservation
  SUM(CASE 
        WHEN RatecodeID IS NULL 
         AND congestion_surcharge IS NULL 
         AND passenger_count IS NULL 
         AND payment_type IS NULL 
         AND store_and_fwd_flag IS NULL 
         AND trip_type IS NULL 
        THEN 1 
        ELSE 0 
      END) AS systematic_null_pattern_rows

FROM `ftw-week-08`.`02_silver`.`green_taxi`;

In [0]:
%sql 
SELECT 
    COUNT(*) AS total_rows, 
    COUNT(DISTINCT source_file) AS source_files, 
 
    -- Trip distance handling by category 
    COUNT_IF(dq_zero_trip_distance = TRUE) AS flagged_zero_trip_distance, 
    COUNT_IF(dq_extreme_trip_distance = TRUE) AS flagged_extreme_trip_distance, 
     
    -- Verify extreme trip distances are nullified 
    COUNT_IF(trip_distance > 1000) AS remaining_extreme_values, 
    COUNT_IF( 
        dq_extreme_trip_distance = TRUE 
        AND trip_distance IS NULL 
    ) AS trip_distance_nullified, 
 
    -- Verify negative trip distances are nullified 
    COUNT_IF( 
        dq_negative_trip_distance = TRUE 
        AND trip_distance IS NULL 
    ) AS negative_trip_distance_nullified, 
 
    -- Out-of-range datetime handling 
    COUNT_IF(dq_out_of_range_datetime = TRUE) AS flagged_out_of_range_datetime, 
 
    -- Invalid trip duration handling 
    COUNT_IF(dq_invalid_trip_duration = TRUE) AS flagged_invalid_trip_duration,

    -- Verify invalid trip durations are nullified
    COUNT_IF(
        dq_invalid_trip_duration = TRUE
        AND trip_duration_minutes IS NOT NULL
    ) AS invalid_duration_with_value,

    COUNT_IF(
        dq_invalid_trip_duration = TRUE
        AND trip_duration_minutes IS NULL
    ) AS invalid_duration_nullified,
 
    -- Systematic NULL pattern preservation 
    COUNT_IF( 
        RatecodeID IS NULL 
        AND congestion_surcharge IS NULL 
        AND passenger_count IS NULL 
        AND payment_type IS NULL 
        AND store_and_fwd_flag IS NULL 
        AND trip_type IS NULL 
    ) AS systematic_null_pattern_rows 
 
FROM `ftw-week-08`.`02_silver`.`green_taxi`;

In [0]:
%sql
-- Show sample rows with data quality issues flagged
SELECT 
  VendorID,
  pickup_hour,
  dropoff_hour,  
  trip_distance,
  fare_amount,
  total_amount,
  dq_zero_trip_distance,
  dq_extreme_trip_distance,
  dq_out_of_range_datetime,
  dq_invalid_trip_duration,
  dq_negative_trip_distance,
  passenger_count,
  RatecodeID,
  payment_type
FROM `ftw-week-08`.`02_silver`.`green_taxi`
WHERE
      dq_zero_trip_distance = TRUE 
   OR dq_extreme_trip_distance = TRUE 
   OR dq_negative_trip_distance = TRUE
   OR dq_out_of_range_datetime = TRUE
   OR dq_invalid_trip_duration = TRUE
LIMIT 10

In [0]:
%sql
-- Verify all required columns are present and _rescued_data and ehail_fee are dropped
DESCRIBE `ftw-week-08`.`02_silver`.`green_taxi`

## 2.5. Categorical Field Validation

Validate categorical/code fields against official NYC TLC Green Taxi mappings:

**RatecodeID** (valid: 1-6, 99):
- 1 = Standard rate
- 2 = JFK
- 3 = Newark
- 4 = Nassau or Westchester
- 5 = Negotiated fare
- 6 = Group ride
- 99 = Unknown (documented special value)

**payment_type** (valid: 0-6):
- 0 = Flex Fare Trip
- 1 = Credit card
- 2 = Cash
- 3 = No charge
- 4 = Dispute
- 5 = Unknown
- 6 = Voided trip

NULL values are counted separately as missing data, not as invalid codes.

In [0]:
-- RatecodeID validation: NULL, valid, and invalid counts
SELECT
    COUNT(*) AS total_rows,
    COUNT_IF(RatecodeID IS NULL) AS null_ratecode,
    COUNT_IF(RatecodeID IN (1, 2, 3, 4, 5, 6, 99)) AS valid_ratecode,
    COUNT_IF(RatecodeID IS NOT NULL AND RatecodeID NOT IN (1, 2, 3, 4, 5, 6, 99)) AS invalid_ratecode,
    
    -- Breakdown by documented code
    COUNT_IF(RatecodeID = 1) AS ratecode_1_standard,
    COUNT_IF(RatecodeID = 2) AS ratecode_2_jfk,
    COUNT_IF(RatecodeID = 3) AS ratecode_3_newark,
    COUNT_IF(RatecodeID = 4) AS ratecode_4_nassau_westchester,
    COUNT_IF(RatecodeID = 5) AS ratecode_5_negotiated,
    COUNT_IF(RatecodeID = 6) AS ratecode_6_group,
    COUNT_IF(RatecodeID = 99) AS ratecode_99_unknown
FROM `ftw-week-08`.`02_silver`.`green_taxi`

In [0]:
-- payment_type validation: NULL, valid, and invalid counts
SELECT
    COUNT(*) AS total_rows,
    COUNT_IF(payment_type IS NULL) AS null_payment_type,
    COUNT_IF(payment_type IN (0, 1, 2, 3, 4, 5, 6)) AS valid_payment_type,
    COUNT_IF(payment_type IS NOT NULL AND payment_type NOT IN (0, 1, 2, 3, 4, 5, 6)) AS invalid_payment_type,
    
    -- Breakdown by documented code
    COUNT_IF(payment_type = 0) AS payment_0_flex_fare_trip,
    COUNT_IF(payment_type = 1) AS payment_1_credit_card,
    COUNT_IF(payment_type = 2) AS payment_2_cash,
    COUNT_IF(payment_type = 3) AS payment_3_no_charge,
    COUNT_IF(payment_type = 4) AS payment_4_dispute,
    COUNT_IF(payment_type = 5) AS payment_5_unknown,
    COUNT_IF(payment_type = 6) AS payment_6_voided
FROM `ftw-week-08`.`02_silver`.`green_taxi`

In [0]:
%sql
SELECT
  COALESCE(b.source_file, s.source_file) AS source_file,
  COALESCE(b.batch_id, s.batch_id) AS batch_id,
  COALESCE(b.bronze_count, 0) AS bronze_count,
  COALESCE(s.silver_count, 0) AS silver_count,
  COALESCE(s.silver_count, 0) - COALESCE(b.bronze_count, 0) AS difference,
  CASE
    WHEN COALESCE(b.bronze_count, 0) = COALESCE(s.silver_count, 0)
    THEN 'PASS'
    ELSE 'FAIL'
  END AS status
FROM (
  SELECT
    source_file,
    batch_id,
    COUNT(*) AS bronze_count
  FROM `ftw-week-08`.`01_bronze`.`green_taxi`
  GROUP BY source_file, batch_id
) b
FULL OUTER JOIN (
  SELECT
    source_file,
    batch_id,
    COUNT(*) AS silver_count
  FROM `ftw-week-08`.`02_silver`.`green_taxi`
  GROUP BY source_file, batch_id
) s
  ON b.source_file = s.source_file
 AND b.batch_id = s.batch_id
ORDER BY source_file, batch_id;

In [0]:
%sql
SELECT
  COUNT(*) FILTER (WHERE lpep_pickup_datetime IS NULL)
    AS missing_pickup_datetime,

  COUNT(*) FILTER (WHERE lpep_dropoff_datetime IS NULL)
    AS missing_dropoff_datetime,

  COUNT(*) FILTER (WHERE PULocationID IS NULL)
    AS missing_pickup_location_id,

  COUNT(*) FILTER (WHERE DOLocationID IS NULL)
    AS missing_dropoff_location_id
FROM `ftw-week-08`.`02_silver`.`green_taxi`;

In [0]:
%sql
SELECT
  VendorID,
  lpep_pickup_datetime,
  lpep_dropoff_datetime,
  store_and_fwd_flag,
  RatecodeID,
  PULocationID,
  DOLocationID,
  passenger_count,
  trip_distance,
  fare_amount,
  extra,
  mta_tax,
  tip_amount,
  tolls_amount,
  improvement_surcharge,
  total_amount,
  payment_type,
  trip_type,
  congestion_surcharge,
  cbd_congestion_fee,
  COUNT(*) AS duplicate_count
FROM `ftw-week-08`.`01_bronze`.`green_taxi`
GROUP BY
  VendorID,
  lpep_pickup_datetime,
  lpep_dropoff_datetime,
  store_and_fwd_flag,
  RatecodeID,
  PULocationID,
  DOLocationID,
  passenger_count,
  trip_distance,
  fare_amount,
  extra,
  mta_tax,
  tip_amount,
  tolls_amount,
  improvement_surcharge,
  total_amount,
  payment_type,
  trip_type,
  congestion_surcharge,
  cbd_congestion_fee
HAVING COUNT(*) > 1
ORDER BY duplicate_count DESC;

In [0]:
-- Final validation summary with Pass/Fail status
WITH validation_results AS (
    SELECT
        -- Row and source reconciliation
        (SELECT COUNT(*)
         FROM `ftw-week-08`.`01_bronze`.`green_taxi`) AS source_count,

        (SELECT COUNT(*)
         FROM `ftw-week-08`.`02_silver`.`green_taxi`) AS silver_count,

        (SELECT COUNT(DISTINCT source_file)
         FROM `ftw-week-08`.`01_bronze`.`green_taxi`) AS source_files,

        (SELECT COUNT(DISTINCT source_file)
         FROM `ftw-week-08`.`02_silver`.`green_taxi`) AS silver_source_files,

        -- Trip distance
        (SELECT COUNT_IF(dq_zero_trip_distance = TRUE)
         FROM `ftw-week-08`.`02_silver`.`green_taxi`) AS zero_trip_distance,

        (SELECT COUNT_IF(dq_extreme_trip_distance = TRUE)
         FROM `ftw-week-08`.`02_silver`.`green_taxi`) AS extreme_trip_distance,

        (SELECT COUNT_IF(
            dq_extreme_trip_distance = TRUE
            AND trip_distance IS NULL
        )
         FROM `ftw-week-08`.`02_silver`.`green_taxi`) AS extreme_distance_nullified,

        (SELECT COUNT_IF(dq_negative_trip_distance = TRUE)
         FROM `ftw-week-08`.`02_silver`.`green_taxi`) AS negative_trip_distance,

        (SELECT COUNT_IF(
            dq_negative_trip_distance = TRUE
            AND trip_distance IS NULL
        )
         FROM `ftw-week-08`.`02_silver`.`green_taxi`) AS negative_distance_nullified,

        (SELECT COUNT_IF(trip_distance > 1000)
         FROM `ftw-week-08`.`02_silver`.`green_taxi`) AS remaining_extreme_distance,

        (SELECT COUNT_IF(trip_distance < 0)
         FROM `ftw-week-08`.`02_silver`.`green_taxi`) AS remaining_negative_distance,

        -- Datetime and duration
        (SELECT COUNT_IF(dq_out_of_range_datetime = TRUE)
         FROM `ftw-week-08`.`02_silver`.`green_taxi`) AS out_of_range_datetime,

        (SELECT COUNT_IF(dq_invalid_trip_duration = TRUE)
         FROM `ftw-week-08`.`02_silver`.`green_taxi`) AS invalid_trip_duration,

        (SELECT COUNT_IF(
            dq_invalid_trip_duration = TRUE
            AND trip_duration_minutes IS NOT NULL
        )
         FROM `ftw-week-08`.`02_silver`.`green_taxi`) AS invalid_duration_with_value,

        -- Missing required fields: profiling only
        (SELECT COUNT_IF(lpep_pickup_datetime IS NULL)
         FROM `ftw-week-08`.`02_silver`.`green_taxi`) AS missing_pickup_datetime,

        (SELECT COUNT_IF(lpep_dropoff_datetime IS NULL)
         FROM `ftw-week-08`.`02_silver`.`green_taxi`) AS missing_dropoff_datetime,

        (SELECT COUNT_IF(PULocationID IS NULL)
         FROM `ftw-week-08`.`02_silver`.`green_taxi`) AS missing_pickup_location_id,

        (SELECT COUNT_IF(DOLocationID IS NULL)
         FROM `ftw-week-08`.`02_silver`.`green_taxi`) AS missing_dropoff_location_id,

        -- Systematic NULL pattern
        (SELECT COUNT_IF(
            RatecodeID IS NULL
            AND congestion_surcharge IS NULL
            AND passenger_count IS NULL
            AND payment_type IS NULL
            AND store_and_fwd_flag IS NULL
            AND trip_type IS NULL
        )
         FROM `ftw-week-08`.`02_silver`.`green_taxi`) AS systematic_null_pattern,

        -- Categorical validation
        (SELECT COUNT_IF(
            RatecodeID IS NOT NULL
            AND RatecodeID NOT IN (1, 2, 3, 4, 5, 6, 99)
        )
         FROM `ftw-week-08`.`02_silver`.`green_taxi`) AS invalid_ratecode,

        (SELECT COUNT_IF(
            payment_type IS NOT NULL
            AND payment_type NOT IN (0, 1, 2, 3, 4, 5, 6)
        )
         FROM `ftw-week-08`.`02_silver`.`green_taxi`) AS invalid_payment_type,

        -- Exact duplicate profiling on original Bronze business columns
        (SELECT COUNT(*)
         FROM (
             SELECT
                 VendorID,
                 lpep_pickup_datetime,
                 lpep_dropoff_datetime,
                 store_and_fwd_flag,
                 RatecodeID,
                 PULocationID,
                 DOLocationID,
                 passenger_count,
                 trip_distance,
                 fare_amount,
                 extra,
                 mta_tax,
                 tip_amount,
                 tolls_amount,
                 improvement_surcharge,
                 total_amount,
                 payment_type,
                 trip_type,
                 congestion_surcharge,
                 cbd_congestion_fee
             FROM `ftw-week-08`.`01_bronze`.`green_taxi`
             GROUP BY
                 VendorID,
                 lpep_pickup_datetime,
                 lpep_dropoff_datetime,
                 store_and_fwd_flag,
                 RatecodeID,
                 PULocationID,
                 DOLocationID,
                 passenger_count,
                 trip_distance,
                 fare_amount,
                 extra,
                 mta_tax,
                 tip_amount,
                 tolls_amount,
                 improvement_surcharge,
                 total_amount,
                 payment_type,
                 trip_type,
                 congestion_surcharge,
                 cbd_congestion_fee
             HAVING COUNT(*) > 1
         )
        ) AS exact_duplicate_groups
),

checks AS (
    SELECT
        'Source row count' AS check_name,
        source_count AS expected,
        silver_count AS actual,
        CASE
            WHEN source_count = silver_count THEN 'PASS'
            ELSE 'FAIL'
        END AS status
    FROM validation_results

    UNION ALL

    SELECT
        'Source file count',
        source_files,
        silver_source_files,
        CASE
            WHEN source_files = silver_source_files THEN 'PASS'
            ELSE 'FAIL'
        END
    FROM validation_results

    UNION ALL

    SELECT
        'Extreme trip distances nullified',
        extreme_trip_distance,
        extreme_distance_nullified,
        CASE
            WHEN extreme_trip_distance = extreme_distance_nullified THEN 'PASS'
            ELSE 'FAIL'
        END
    FROM validation_results

    UNION ALL

    SELECT
        'Remaining trip distance > 1000',
        0,
        remaining_extreme_distance,
        CASE
            WHEN remaining_extreme_distance = 0 THEN 'PASS'
            ELSE 'FAIL'
        END
    FROM validation_results

    UNION ALL

    SELECT
        'Negative trip distances nullified',
        negative_trip_distance,
        negative_distance_nullified,
        CASE
            WHEN negative_trip_distance = negative_distance_nullified THEN 'PASS'
            ELSE 'FAIL'
        END
    FROM validation_results

    UNION ALL

    SELECT
        'Remaining negative trip distances',
        0,
        remaining_negative_distance,
        CASE
            WHEN remaining_negative_distance = 0 THEN 'PASS'
            ELSE 'FAIL'
        END
    FROM validation_results

    UNION ALL

    SELECT
        'Out-of-range datetime flagged',
        19,
        out_of_range_datetime,
        CASE
            WHEN out_of_range_datetime = 19 THEN 'PASS'
            ELSE 'FAIL'
        END
    FROM validation_results

    UNION ALL

    SELECT
        'Invalid trip duration flagged',
        1,
        invalid_trip_duration,
        CASE
            WHEN invalid_trip_duration = 1 THEN 'PASS'
            ELSE 'FAIL'
        END
    FROM validation_results

    UNION ALL

    SELECT
        'Invalid durations produce NULL',
        0,
        invalid_duration_with_value,
        CASE
            WHEN invalid_duration_with_value = 0 THEN 'PASS'
            ELSE 'FAIL'
        END
    FROM validation_results

    UNION ALL

    SELECT
        'Systematic NULL pattern',
        18754,
        systematic_null_pattern,
        CASE
            WHEN systematic_null_pattern = 18754 THEN 'PASS'
            ELSE 'FAIL'
        END
    FROM validation_results

    UNION ALL

    SELECT
        'Invalid RatecodeID',
        0,
        invalid_ratecode,
        CASE
            WHEN invalid_ratecode = 0 THEN 'PASS'
            ELSE 'FAIL'
        END
    FROM validation_results

    UNION ALL

    SELECT
        'Invalid payment_type',
        0,
        invalid_payment_type,
        CASE
            WHEN invalid_payment_type = 0 THEN 'PASS'
            ELSE 'FAIL'
        END
    FROM validation_results

    UNION ALL

    SELECT
        'Exact duplicate groups',
        0,
        exact_duplicate_groups,
        CASE
            WHEN exact_duplicate_groups = 0 THEN 'PASS'
            ELSE 'FAIL'
        END
    FROM validation_results
)

SELECT *
FROM checks
ORDER BY
    CASE check_name
        WHEN 'Source row count' THEN 1
        WHEN 'Source file count' THEN 2
        WHEN 'Extreme trip distances nullified' THEN 3
        WHEN 'Remaining trip distance > 1000' THEN 4
        WHEN 'Negative trip distances nullified' THEN 5
        WHEN 'Remaining negative trip distances' THEN 6
        WHEN 'Out-of-range datetime flagged' THEN 7
        WHEN 'Invalid trip duration flagged' THEN 8
        WHEN 'Invalid durations produce NULL' THEN 9
        WHEN 'Systematic NULL pattern' THEN 10
        WHEN 'Invalid RatecodeID' THEN 11
        WHEN 'Invalid payment_type' THEN 12
        WHEN 'Exact duplicate groups' THEN 13
    END;

## Validation Summary

The final Green Taxi validation confirms that the Silver table preserved all 133,367 source rows across 3 source files. Extreme and negative trip distances were nullified as required, while zero distances were retained. Datetime, trip-duration, systematic NULL-pattern, and categorical validations were also evaluated.

Additional validation covers:
- Missing pickup/dropoff timestamps
- Missing pickup/dropoff LocationIDs
- Bronze-to-Silver reconciliation by `source_file` and `batch_id`
- Exact-duplicate profiling using original Bronze business columns
- Flagged-record sampling, including negative trip distances